# YOLOv5s vs YOLOv8s — Comparaison Améliorée sur Pascal VOC 2007

Ce notebook compare les performances de **YOLOv5s** et **YOLOv8s** sur le dataset Pascal VOC 2007.
Chaque run est sauvegardé dans `results/ameliored/<timestamp>/` pour garder un historique complet.

**Améliorations :**
- Résultats horodatés dans `results/ameliored/YYYYMMDD_HHMMSS/`
- Augmentations avancées : Mixup, Copy-Paste, rotation, perspective
- Optimiseur AdamW + Cosine LR decay + Warmup
- Fine-tuning YOLOv8 en 2 phases (freeze backbone → unfreeze)
- `close_mosaic` pour stabiliser la fin d'entraînement
- Analyse de la distribution des classes
- Analyse mAP par classe
- Courbe F1-Score
- Export ONNX pour la production

18. Tableau récapitulatif final

**Plan :**17. Export ONNX

1. Téléchargement et extraction du dataset VOC16. Vitesse d'inférence (ms/image & FPS)

2. Organisation dans le répertoire `data/`15. Analyse mAP par classe

3. Conversion des annotations VOC → format YOLO14. Courbe F1-Score

4. Analyse de la distribution des classes13. Précision & Rappel

5. Création des fichiers de configuration12. Courbes de Loss

6. Installation des dépendances11. Comparaison mAP (mAP50 & mAP50-95)

7. Configuration GPU & répertoires horodatés10. Chargement des résultats

8. Entraînement YOLOv5s (augmentations avancées + cosine LR)9. Entraînement YOLOv8s (AdamW + 2 phases)

## 1. Téléchargement et extraction du dataset Pascal VOC

In [3]:
import os
import urllib.request
import tarfile
import zipfile
from pathlib import Path

# Répertoires de base
BASE_DIR = Path(".")
DATA_DIR = BASE_DIR / "data"
VOC_DIR = DATA_DIR / "VOCdevkit"
DATA_DIR.mkdir(exist_ok=True)

# URLs officielles Pascal VOC 2007 (~450 MB train+val+test)
VOC_URLS = {
    "trainval": "http://host.robots.ox.ac.uk/pascal/VOC/voc2007/VOCtrainval_06-Nov-2007.tar",
    "test":     "http://host.robots.ox.ac.uk/pascal/VOC/voc2007/VOCtest_06-Nov-2007.tar",
}

def download_with_progress(url: str, dest: Path):
    """Télécharge un fichier avec affichage de la progression."""
    filename = dest / Path(url).name
    if filename.exists():
        print(f"[SKIP] {filename.name} déjà téléchargé.")
        return filename

    print(f"[DOWNLOAD] {url}")
    downloaded = [0]
    total = [0]

    def reporthook(block_num, block_size, total_size):
        downloaded[0] = block_num * block_size
        total[0] = total_size
        if total_size > 0:
            pct = min(downloaded[0] / total_size * 100, 100)
            print(f"\r  {pct:.1f}% ({downloaded[0] / 1e6:.1f} / {total_size / 1e6:.1f} MB)", end="")

    urllib.request.urlretrieve(url, filename, reporthook)
    print(f"\n[OK] Sauvegardé dans {filename}")
    return filename


for split, url in VOC_URLS.items():
    tar_path = download_with_progress(url, DATA_DIR)
    if not (DATA_DIR / "VOCdevkit").exists():
        print(f"[EXTRACT] {tar_path.name} ...")
        with tarfile.open(tar_path) as t:
            t.extractall(DATA_DIR)
        print("[OK] Extraction terminée.")

print("\n[OK] Dataset VOC 2007 prêt dans", VOC_DIR)

[SKIP] VOCtrainval_06-Nov-2007.tar déjà téléchargé.
[SKIP] VOCtest_06-Nov-2007.tar déjà téléchargé.

[OK] Dataset VOC 2007 prêt dans data/VOCdevkit


## 2. Organisation du dataset dans le répertoire `data/`

In [4]:
import shutil
import xml.etree.ElementTree as ET

VOC2007_DIR = VOC_DIR / "VOC2007"
IMG_SRC     = VOC2007_DIR / "JPEGImages"
ANN_SRC     = VOC2007_DIR / "Annotations"
SETS_SRC    = VOC2007_DIR / "ImageSets" / "Main"

# Répertoires cibles
IMAGES_DIR = DATA_DIR / "images"
LABELS_DIR = DATA_DIR / "labels"

for split in ("train", "val", "test"):
    (IMAGES_DIR / split).mkdir(parents=True, exist_ok=True)
    (LABELS_DIR / split).mkdir(parents=True, exist_ok=True)

# Mapping split files VOC → nos splits
SPLIT_MAP = {
    "train": SETS_SRC / "trainval.txt",
    "val":   SETS_SRC / "val.txt",
    "test":  SETS_SRC / "test.txt",
}

split_ids = {}
for split, txt_file in SPLIT_MAP.items():
    if not txt_file.exists():
        # fallback: utiliser train.txt si val.txt absent
        alt = SETS_SRC / f"{split}.txt"
        txt_file = alt if alt.exists() else list(SETS_SRC.glob("*.txt"))[0]
    with open(txt_file) as f:
        ids = [line.strip().split()[0] for line in f if line.strip()]
    split_ids[split] = ids
    print(f"  {split}: {len(ids)} images")

# Copier les images dans les bons dossiers
for split, ids in split_ids.items():
    for img_id in ids:
        src = IMG_SRC / f"{img_id}.jpg"
        dst = IMAGES_DIR / split / f"{img_id}.jpg"
        if src.exists() and not dst.exists():
            shutil.copy2(src, dst)

print("[OK] Images organisées dans data/images/{train,val,test}/")

  train: 5011 images
  val: 2510 images
  test: 2510 images
[OK] Images organisées dans data/images/{train,val,test}/


## 3. Conversion des annotations VOC (XML) → format YOLO

In [5]:
import os
import xml.etree.ElementTree as ET
from concurrent.futures import ProcessPoolExecutor, as_completed
from pathlib import Path

VOC_CLASSES = [
    "aeroplane", "bicycle", "bird", "boat", "bottle",
    "bus", "car", "cat", "chair", "cow",
    "diningtable", "dog", "horse", "motorbike", "person",
    "pottedplant", "sheep", "sofa", "train", "tvmonitor"
]
CLASS2IDX = {c: i for i, c in enumerate(VOC_CLASSES)}


def _convert_and_write(args):
    """Convertit un XML VOC → YOLO et écrit le .txt — exécuté en parallèle."""
    xml_path_str, label_path_str, class2idx = args
    xml_path   = Path(xml_path_str)
    label_path = Path(label_path_str)
    if not xml_path.exists() or label_path.exists():
        return 0
    try:
        tree = ET.parse(xml_path)
        root = tree.getroot()
        size = root.find("size")
        w = int(size.find("width").text)
        h = int(size.find("height").text)
        lines = []
        for obj in root.findall("object"):
            cls_name = obj.find("name").text.strip()
            if cls_name not in class2idx:
                continue
            diff = obj.find("difficult")
            if diff is not None and int(diff.text) == 1:
                continue
            cls_id = class2idx[cls_name]
            bb = obj.find("bndbox")
            xmin = float(bb.find("xmin").text)
            ymin = float(bb.find("ymin").text)
            xmax = float(bb.find("xmax").text)
            ymax = float(bb.find("ymax").text)
            lines.append(
                f"{cls_id} {(xmin+xmax)/2/w:.6f} {(ymin+ymax)/2/h:.6f} "
                f"{(xmax-xmin)/w:.6f} {(ymax-ymin)/h:.6f}"
            )
        label_path.write_text("\n".join(lines))
        return 1
    except Exception:
        return 0


# ── Construction de la liste de tâches ───────────────────────────────────────
tasks = [
    (str(ANN_SRC / f"{img_id}.xml"), str(LABELS_DIR / split / f"{img_id}.txt"), CLASS2IDX)
    for split, ids in split_ids.items()
    for img_id in ids
]

# ── Conversion parallèle — tâche one-shot, tous les vCPUs disponibles ────────
NUM_WORKERS = os.cpu_count()   # 16 vCPUs → tous utilisés (tâche courte)
print(f"[INFO] {len(tasks)} fichiers à convertir sur {NUM_WORKERS} workers ...")

total_converted, done = 0, 0
with ProcessPoolExecutor(max_workers=NUM_WORKERS) as executor:
    futures = [executor.submit(_convert_and_write, t) for t in tasks]
    for fut in as_completed(futures):
        total_converted += fut.result()
        done += 1
        if done % 1000 == 0 or done == len(tasks):
            print(f"\r  {done}/{len(tasks)} ({done/len(tasks)*100:.0f}%)", end="", flush=True)

print(f"\n[OK] {total_converted} fichiers convertis en format YOLO.")

[INFO] 10031 fichiers à convertir sur 96 workers ...
  10031/10031 (100%)
[OK] 0 fichiers convertis en format YOLO.


## 4. Analyse de la distribution des classes

In [ ]:
import xml.etree.ElementTree as ET
from collections import Counter
import matplotlib.pyplot as plt
import numpy as np

# Compter toutes les instances par classe dans le set train
counts = Counter()
for img_id in split_ids["train"]:
    xml_file = ANN_SRC / f"{img_id}.xml"
    if not xml_file.exists():
        continue
    tree = ET.parse(xml_file)
    for obj in tree.getroot().findall("object"):
        diff = obj.find("difficult")
        if diff is not None and int(diff.text) == 1:
            continue
        name = obj.find("name").text.strip()
        if name in CLASS2IDX:
            counts[name] += 1

sorted_classes = [c for c, _ in counts.most_common()]
sorted_counts  = [counts[c] for c in sorted_classes]
total = sum(sorted_counts)
max_count = max(sorted_counts)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle("Distribution des Classes — Pascal VOC 2007 (Train)", fontsize=14, fontweight="bold")

colors_bar = plt.cm.RdYlGn(np.linspace(0.2, 0.9, len(sorted_classes)))
ax = axes[0]
bars = ax.barh(sorted_classes, sorted_counts, color=colors_bar)
ax.set_xlabel("Nombre d'instances")
ax.set_title("Instances par classe (train)")
ax.bar_label(bars, fmt="%d", padding=3, fontsize=8)
ax.invert_yaxis()
ax.grid(axis="x", alpha=0.3)

ax2 = axes[1]
ratios = [c / max_count for c in sorted_counts]
bars2 = ax2.barh(sorted_classes, ratios, color=plt.cm.RdYlGn(ratios))
ax2.set_xlabel("Ratio (vs classe dominante)")
ax2.set_title("Ratio de déséquilibre")
ax2.bar_label(bars2, fmt="%.2f", padding=3, fontsize=8)
ax2.invert_yaxis()
ax2.axvline(x=0.3, color="red", linestyle="--", alpha=0.6, label="Seuil critique (0.3)")
ax2.legend()
ax2.grid(axis="x", alpha=0.3)

plt.tight_layout()
plt.savefig(BASE_DIR / "class_distribution.png", dpi=150)
plt.show()

underrepresented = [c for c, n in counts.items() if n / max_count < 0.3]
print(f"[INFO] Classes sous-représentées (ratio < 0.3) : {underrepresented}")
print(f"[INFO] Total instances train : {total:,}")
print("[OK] Graphique sauvegardé : class_distribution.png")

## 5. Création des fichiers de configuration `data.yaml`

In [10]:
import yaml

ABS_DATA = DATA_DIR.resolve()

# Utiliser des chemins absolus pour la portabilité
voc_yaml_content = {
    "path": str(ABS_DATA),
    "train": str(ABS_DATA / "images" / "train"),
    "val":   str(ABS_DATA / "images" / "val"),
    "test":  str(ABS_DATA / "images" / "test"),
    "nc": len(VOC_CLASSES),
    "names": VOC_CLASSES,
}

# Fichier unique partagé par YOLOv5 et YOLOv8
YAML_PATH = DATA_DIR / "voc.yaml"
with open(YAML_PATH, "w") as f:
    yaml.dump(voc_yaml_content, f, default_flow_style=False, allow_unicode=True)

print(f"[OK] Fichier de configuration créé : {YAML_PATH}")
print(yaml.dump(voc_yaml_content, default_flow_style=False))

[OK] Fichier de configuration créé : data/voc.yaml
names:
- aeroplane
- bicycle
- bird
- boat
- bottle
- bus
- car
- cat
- chair
- cow
- diningtable
- dog
- horse
- motorbike
- person
- pottedplant
- sheep
- sofa
- train
- tvmonitor
nc: 20
path: /workspace/data
test: /workspace/data/images/test
train: /workspace/data/images/train
val: /workspace/data/images/val



## 6. Installation des dépendances YOLOv5 et YOLOv8

In [5]:
import subprocess, sys

# ── YOLOv8 via Ultralytics (pip) ──────────────────────────────────────────────
subprocess.run([sys.executable, "-m", "pip", "install", "ultralytics", "-q"], check=True)
print("[OK] Ultralytics (YOLOv8) installé.")

# ── YOLOv5 — cloner le dépôt officiel si nécessaire ──────────────────────────
YOLOV5_DIR = BASE_DIR / "yolov5"
if not YOLOV5_DIR.exists():
    subprocess.run(
        ["git", "clone", "https://github.com/ultralytics/yolov5.git", str(YOLOV5_DIR)],
        check=True,
    )
    print("[OK] Dépôt YOLOv5 cloné.")
else:
    print("[SKIP] Dépôt YOLOv5 déjà présent.")

# Installer les dépendances YOLOv5
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r", str(YOLOV5_DIR / "requirements.txt"), "-q"],
    check=True,
)
print("[OK] Dépendances YOLOv5 installées.")


[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: python -m pip install --upgrade pip


[OK] Ultralytics (YOLOv8) installé.


Cloning into 'yolov5'...


[OK] Dépôt YOLOv5 cloné.
[OK] Dépendances YOLOv5 installées.



[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: python -m pip install --upgrade pip


## 7. Initialisation du modèle YOLOv8

In [7]:
from ultralytics import YOLO

# Charger YOLOv8n pré-entraîné (nano — le plus rapide)
yolov8_model = YOLO("yolov8n.pt")
print(f"[INFO] YOLOv8n chargé — paramètres : {sum(p.numel() for p in yolov8_model.model.parameters()):,}")
print(f"[INFO] YOLOv8  — modèle : yolov8n  | epochs : {EPOCHS} | batch : {BATCH} | img : {IMG_SZ}")
print(f"[INFO] Répertoire de sorties YOLOv8 : {YV8_RUN_DIR}")

WARNING ⚠️ user config directory '/root/.config/Ultralytics' is not writable, using '/tmp/Ultralytics'. Set YOLO_CONFIG_DIR to override.
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/tmp/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
[INFO] YOLOv8n chargé — paramètres : 3,157,200
[INFO] YOLOv8  — modèle : yolov8n  | epochs : 100 | batch : 128 | img : 640
[INFO] Répertoire de sorties YOLOv8 : runs/yolov8


## 8. Entraînement YOLOv5s — Augmentations Avancées + Cosine LR

In [ ]:
import subprocess, sys, yaml as _yaml
from pathlib import Path

# ── Fichier hyp avec augmentations avancées + cls_pw pour classes rares ──────
hyp_advanced = {
    "lr0": 0.005, "lrf": 0.1, "momentum": 0.937, "weight_decay": 0.0005,
    "warmup_epochs": 3.0, "warmup_momentum": 0.8, "warmup_bias_lr": 0.1,
    "box": 0.05, "cls": 0.5, "cls_pw": 1.5,  # cls_pw > 1 → pénalise classes rares
    "obj": 1.0, "obj_pw": 1.0, "iou_t": 0.20, "anchor_t": 4.0, "fl_gamma": 0.0,
    "hsv_h": 0.015, "hsv_s": 0.7, "hsv_v": 0.4,
    "degrees": 10.0, "translate": 0.1, "scale": 0.5, "shear": 2.0,
    "perspective": 0.0001, "flipud": 0.0, "fliplr": 0.5,
    "mosaic": 1.0, "mixup": 0.1, "copy_paste": 0.1,  # augmentations avancées
}
HYP_PATH = RUN_ROOT / "hyp_advanced.yaml"
with open(HYP_PATH, "w") as f:
    _yaml.dump(hyp_advanced, f)
print(f"[OK] Fichier hyp avancé : {HYP_PATH}")


def _run_yolov5() -> int:
    cmd = [
        sys.executable, str(YOLOV5_DIR / "train.py"),
        "--img",     str(IMG_SZ),
        "--batch",   str(BATCH),
        "--epochs",  str(EPOCHS),
        "--data",    str(YAML_PATH.resolve()),
        "--weights", "yolov5s.pt",
        "--hyp",     str(HYP_PATH.resolve()),
        "--cos-lr",
        "--project", str(YV5_RUN_DIR),
        "--name",    "voc",
        "--exist-ok",
        "--device",  DEVICE,
        "--workers", str(WORKERS),
    ]
    with subprocess.Popen(
        cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1, encoding="utf-8", errors="replace",
    ) as proc:
        for line in proc.stdout:
            print(line, end="", flush=True)
    return proc.returncode


YV5_RESULT_DIR = train_model("YOLOv5s", _run_yolov5, YV5_RUN_DIR / "voc")


──────────────────────────────────────────────────────────────
  🚀  Démarrage entraînement  YOLOv5s
──────────────────────────────────────────────────────────────
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
train: weights=yolov5s.pt, cfg=, data=/workspace/data/voc.yaml, hyp=yolov5/data/hyps/hyp.scratch-low.yaml, epochs=100, batch_size=128, imgsz=640, rect=False, resume=False, nosave=False, noval=False, noautoanchor=False, noplots=False, evolve=None, evolve_population=yolov5/data/hyps, resume_evolve=None, bucket=, cache=None, image_weights=False, device=0, multi_scale=False, single_cls=False, optimizer=SGD, sync_bn=False, workers=8, project=runs/yolov5, name=voc, exist_ok=True, quad=False, cos_lr=False, label_smoothing=0.0,

## 9. Entraînement YOLOv8s — AdamW + Mixup + Copy-Paste + 2 Phases

In [ ]:
from ultralytics import YOLO


def _run_yolov8() -> int:
    # ── Phase 1 : backbone gelé — apprentissage de la tête (10 epochs) ────────
    print("\n[PHASE 1] Freeze backbone (10 epochs, lr=0.001) ...")
    model = YOLO("yolov8s.pt")
    model.train(
        data=str(YAML_PATH.resolve()), epochs=10, batch=BATCH, imgsz=IMG_SZ,
        device=DEVICE, project=str(YV8_RUN_DIR), name="voc_phase1", exist_ok=True,
        workers=WORKERS, optimizer="AdamW", lr0=0.001, freeze=10, verbose=False,
    )

    # ── Phase 2 : réseau complet dégelé avec augmentations avancées ───────────
    print("\n[PHASE 2] Fine-tuning complet (90 epochs, augmentations avancées) ...")
    best_phase1 = YV8_RUN_DIR / "voc_phase1" / "weights" / "best.pt"
    model2 = YOLO(str(best_phase1))
    model2.train(
        data=str(YAML_PATH.resolve()), epochs=EPOCHS - 10, batch=BATCH,
        imgsz=IMG_SZ, device=DEVICE, project=str(YV8_RUN_DIR), name="voc",
        exist_ok=True, workers=WORKERS,
        optimizer="AdamW", lr0=0.005, lrf=0.1, warmup_epochs=3,
        cos_lr=True, freeze=0,
        mixup=0.1, copy_paste=0.1, degrees=10.0, perspective=0.001,
        mosaic=1.0, close_mosaic=10,
        verbose=True,
    )
    return 0


YV8_RESULT_DIR = train_model("YOLOv8s", _run_yolov8, YV8_RUN_DIR / "voc")

[OK] YOLOv5 results : runs/yolov5/voc  —  existe : True

──────────────────────────────────────────────────────────────
  🚀  Démarrage entraînement  YOLOv8s
──────────────────────────────────────────────────────────────
Ultralytics 8.4.89 🚀 Python-3.11.10 torch-2.4.1+cu124 CUDA:0 (NVIDIA A40, 45498MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=128, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/workspace/data/voc.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, m

## 10. Chargement des résultats d'entraînement

In [ ]:
import pandas as pd

# Chemins relatifs au run courant (pas de chemin hardcodé)
YV5_RESULT_DIR = YV5_RUN_DIR / "voc"
YV8_RESULT_DIR = YV8_RUN_DIR / "voc"

df_v5 = pd.read_csv(YV5_RESULT_DIR / "results.csv")
df_v5.columns = df_v5.columns.str.strip()
print("Colonnes YOLOv5 :", list(df_v5.columns))

df_v8 = pd.read_csv(YV8_RESULT_DIR / "results.csv")
df_v8.columns = df_v8.columns.str.strip()
print("Colonnes YOLOv8 :", list(df_v8.columns))

df_v5["model"] = "YOLOv5s"
df_v8["model"] = "YOLOv8s"

print(f"\n[OK] YOLOv5 : {len(df_v5)} epochs | répertoire : {YV5_RESULT_DIR}")
print(f"[OK] YOLOv8 : {len(df_v8)} epochs | répertoire : {YV8_RESULT_DIR}")


print("[OK] Colonnes mappées.")

def get_col(df, candidates):epochs_v8 = range(1, len(df_v8) + 1)

    for c in candidates:epochs_v5 = range(1, len(df_v5) + 1)

        if c in df.columns:

            return cV8_REC     = get_col(df_v8, ["metrics/recall(B)",    "metrics/recall"])

    raise KeyError(f"Aucune colonne parmi {candidates}\nDisponibles : {list(df.columns)}")V8_PREC    = get_col(df_v8, ["metrics/precision(B)", "metrics/precision"])

V5_REC     = get_col(df_v5, ["metrics/recall",       "metrics/recall(B)"])

V5_MAP50   = get_col(df_v5, ["metrics/mAP_0.5",      "metrics/mAP50(B)"])V5_PREC    = get_col(df_v5, ["metrics/precision",    "metrics/precision(B)"])

V5_MAP5095 = get_col(df_v5, ["metrics/mAP_0.5:0.95", "metrics/mAP50-95(B)"])V8_MAP5095 = get_col(df_v8, ["metrics/mAP50-95(B)",   "metrics/mAP_0.5:0.95"])
V8_MAP50   = get_col(df_v8, ["metrics/mAP50(B)",      "metrics/mAP_0.5"])

Colonnes YOLOv5 : ['epoch', 'train/box_loss', 'train/obj_loss', 'train/cls_loss', 'metrics/precision', 'metrics/recall', 'metrics/mAP_0.5', 'metrics/mAP_0.5:0.95', 'val/box_loss', 'val/obj_loss', 'val/cls_loss', 'x/lr0', 'x/lr1', 'x/lr2']
Colonnes YOLOv8 : ['epoch', 'time', 'train/box_loss', 'train/cls_loss', 'train/dfl_loss', 'metrics/precision(B)', 'metrics/recall(B)', 'metrics/mAP50(B)', 'metrics/mAP50-95(B)', 'val/box_loss', 'val/cls_loss', 'val/dfl_loss', 'lr/pg0', 'lr/pg1', 'lr/pg2']

[OK] YOLOv5 : 100 epochs chargées.
[OK] YOLOv8 : 100 epochs chargées.


## 11. Comparaison mAP (mAP50 et mAP50-95)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Comparaison mAP — YOLOv5s vs YOLOv8s (Pascal VOC 2007)", fontsize=14, fontweight="bold")

ax = axes[0]
ax.plot(epochs_v5, df_v5[V5_MAP50], label="YOLOv5s", color="#2196F3", linewidth=2)
ax.plot(epochs_v8, df_v8[V8_MAP50], label="YOLOv8s", color="#FF5722", linewidth=2)
ax.axhline(y=df_v5[V5_MAP50].max(), color="#2196F3", linestyle="--", alpha=0.4,
           label=f"V5 max={df_v5[V5_MAP50].max()*100:.1f}%")
ax.axhline(y=df_v8[V8_MAP50].max(), color="#FF5722", linestyle="--", alpha=0.4,
           label=f"V8 max={df_v8[V8_MAP50].max()*100:.1f}%")
ax.set_title("mAP @ IoU 0.50")
ax.set_xlabel("Epoch")
ax.set_ylabel("mAP50")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))

ax = axes[1]
ax.plot(epochs_v5, df_v5[V5_MAP5095], label="YOLOv5s", color="#2196F3", linewidth=2)
ax.plot(epochs_v8, df_v8[V8_MAP5095], label="YOLOv8s", color="#FF5722", linewidth=2)
ax.set_title("mAP @ IoU 0.50 : 0.95")
ax.set_xlabel("Epoch")
ax.set_ylabel("mAP50-95")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))

plt.tight_layout()
plt.savefig(RUN_ROOT / "map_comparison.png", dpi=150)
plt.show()
print(f"[OK] Graphique sauvegardé dans {RUN_ROOT}/map_comparison.png")

<Figure size 1400x500 with 2 Axes>

[OK] Graphique sauvegardé : map_comparison.png


## 12. Comparaison des courbes de Loss (Box / Cls / Obj)

In [ ]:
# Colonnes de loss — YOLOv5 et YOLOv8 ont des noms légèrement différents
LOSS_MAP = {
    "Box Loss (train)": {
        "v5": get_col(df_v5, ["train/box_loss", "train/box_om"]),
        "v8": get_col(df_v8, ["train/box_loss", "train/box_om"]),
    },
    "Cls Loss (train)": {
        "v5": get_col(df_v5, ["train/cls_loss", "train/cls_om"]),
        "v8": get_col(df_v8, ["train/cls_loss", "train/cls_om"]),
    },
    "Box Loss (val)": {
        "v5": get_col(df_v5, ["val/box_loss", "val/box_om"]),
        "v8": get_col(df_v8, ["val/box_loss", "val/box_om"]),
    },
    "Cls Loss (val)": {
        "v5": get_col(df_v5, ["val/cls_loss", "val/cls_om"]),
        "v8": get_col(df_v8, ["val/cls_loss", "val/cls_om"]),
    },
}

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Courbes de Loss — YOLOv5s vs YOLOv8s", fontsize=14, fontweight="bold")

for ax, (title, cols) in zip(axes.flat, LOSS_MAP.items()):
    ax.plot(epochs_v5, df_v5[cols["v5"]], label="YOLOv5s", color="#2196F3", linewidth=2)
    ax.plot(epochs_v8, df_v8[cols["v8"]], label="YOLOv8n", color="#FF5722", linewidth=2)
    ax.set_title(title)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(RUN_ROOT / "loss_comparison.png", dpi=150)
plt.show()
print(f"[OK] Graphique sauvegardé dans {RUN_ROOT}/loss_comparison.png")

<Figure size 1400x1000 with 4 Axes>

[OK] Graphique sauvegardé : loss_comparison.png


## 13. Comparaison Précision & Rappel

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Précision & Rappel — YOLOv5s vs YOLOv8s", fontsize=14, fontweight="bold")

ax = axes[0]
ax.plot(epochs_v5, df_v5[V5_PREC], label="YOLOv5s", color="#2196F3", linewidth=2)
ax.plot(epochs_v8, df_v8[V8_PREC], label="YOLOv8s", color="#FF5722", linewidth=2)
ax.set_title("Précision (Precision)")
ax.set_xlabel("Epoch")
ax.set_ylabel("Précision")
ax.legend()
ax.grid(True, alpha=0.3)
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))

ax = axes[1]
ax.plot(epochs_v5, df_v5[V5_REC], label="YOLOv5s", color="#2196F3", linewidth=2)
ax.plot(epochs_v8, df_v8[V8_REC], label="YOLOv8s", color="#FF5722", linewidth=2)
ax.set_title("Rappel (Recall)")
ax.set_xlabel("Epoch")
ax.set_ylabel("Rappel")
ax.legend()
ax.grid(True, alpha=0.3)
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))

plt.tight_layout()
plt.savefig(RUN_ROOT / "precision_recall_comparison.png", dpi=150)
plt.show()
print(f"[OK] Graphique sauvegardé dans {RUN_ROOT}/precision_recall_comparison.png")

<Figure size 1400x500 with 2 Axes>

[OK] Graphique sauvegardé : precision_recall_comparison.png


## 14. Courbe F1-Score

In [ ]:
import numpy as np

def compute_f1(df, prec_col, rec_col):
    p = df[prec_col].values
    r = df[rec_col].values
    denom = p + r
    denom[denom == 0] = 1e-9
    return 2 * p * r / denom

f1_v5 = compute_f1(df_v5, V5_PREC, V5_REC)
f1_v8 = compute_f1(df_v8, V8_PREC, V8_REC)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(epochs_v5, f1_v5, label=f"YOLOv5s  (max={f1_v5.max():.3f})", color="#2196F3", linewidth=2)
ax.plot(epochs_v8, f1_v8, label=f"YOLOv8s  (max={f1_v8.max():.3f})", color="#FF5722", linewidth=2)
ax.axhline(y=f1_v5.max(), color="#2196F3", linestyle="--", alpha=0.4)
ax.axhline(y=f1_v8.max(), color="#FF5722", linestyle="--", alpha=0.4)

best_epoch_v5 = int(np.argmax(f1_v5)) + 1
best_epoch_v8 = int(np.argmax(f1_v8)) + 1
ax.axvline(x=best_epoch_v5, color="#2196F3", linestyle=":", alpha=0.5,
           label=f"V5 meilleure epoch : {best_epoch_v5}")
ax.axvline(x=best_epoch_v8, color="#FF5722", linestyle=":", alpha=0.5,
           label=f"V8 meilleure epoch : {best_epoch_v8}")

ax.set_title("Courbe F1-Score — YOLOv5s vs YOLOv8s", fontsize=14, fontweight="bold")
ax.set_xlabel("Epoch")
ax.set_ylabel("F1-Score")
ax.legend()
ax.grid(True, alpha=0.3)
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
plt.tight_layout()
plt.savefig(RUN_ROOT / "f1_comparison.png", dpi=150)
plt.show()
print(f"[INFO] YOLOv5s — F1 max : {f1_v5.max():.4f} à l'epoch {best_epoch_v5}")
print(f"[INFO] YOLOv8s — F1 max : {f1_v8.max():.4f} à l'epoch {best_epoch_v8}")
print(f"[OK] Graphique sauvegardé dans {RUN_ROOT}/f1_comparison.png")

## 15. Analyse mAP par Classe

## 17. Export ONNX pour la production

In [ ]:
from ultralytics import YOLO
import subprocess, sys, time

print("=" * 55)
print("  Export des modèles en ONNX (optimisé pour production)")
print("=" * 55)

# ── YOLOv8 → ONNX ────────────────────────────────────────────────────────────
model_v8 = YOLO(str(YV8_RESULT_DIR / "weights" / "best.pt"))
t0 = time.time()
export_v8 = model_v8.export(format="onnx", imgsz=IMG_SZ, dynamic=True, simplify=True, opset=17)
print(f"[OK] YOLOv8s exporté en ONNX ({time.time()-t0:.1f}s) → {export_v8}")

# ── YOLOv5 → ONNX ────────────────────────────────────────────────────────────
t0 = time.time()
cmd_export = [
    sys.executable, str(YOLOV5_DIR / "export.py"),
    "--weights", str(YV5_RESULT_DIR / "weights" / "best.pt"),
    "--include", "onnx", "--imgsz", str(IMG_SZ),
    "--simplify", "--dynamic", "--device", DEVICE,
]
result = subprocess.run(cmd_export, capture_output=True, text=True)
if result.returncode == 0:
    print(f"[OK] YOLOv5s exporté en ONNX ({time.time()-t0:.1f}s)")
else:
    print(f"[WARN] Export YOLOv5 ONNX : {result.stderr[-300:]}")

# ── Benchmark ONNX Runtime ────────────────────────────────────────────────────
try:
    import onnxruntime as ort
    import numpy as np
    v8_onnx_path = str(YV8_RESULT_DIR / "weights" / "best.onnx")
    sess = ort.InferenceSession(v8_onnx_path,
                                providers=["CUDAExecutionProvider", "CPUExecutionProvider"])
    input_name = sess.get_inputs()[0].name
    dummy      = np.random.rand(1, 3, IMG_SZ, IMG_SZ).astype(np.float32)
    for _ in range(5): sess.run(None, {input_name: dummy})  # warm-up

    times_onnx = []
    for _ in range(50):
        t0 = time.perf_counter()
        sess.run(None, {input_name: dummy})
        times_onnx.append((time.perf_counter() - t0) * 1000)
    ms_onnx  = float(np.mean(times_onnx))
    fps_onnx = 1000.0 / ms_onnx
    speedup  = ms_v8 / ms_onnx
    print(f"\n[ONNX Runtime] YOLOv8s : {ms_onnx:.2f} ms/image — {fps_onnx:.1f} FPS")
    print(f"[INFO] Accélération ONNX vs PyTorch : x{speedup:.2f}")
except ImportError:
    print("[WARN] onnxruntime non installé — pip install onnxruntime-gpu")

In [ ]:
import glob, numpy as np, time, torch
from ultralytics import YOLO
from pathlib import Path

YOLOV5_DIR = BASE_DIR / "yolov5"

test_images = sorted(glob.glob(str(IMAGES_DIR / "test" / "*.jpg")))[:50]
if not test_images:
    test_images = sorted(glob.glob(str(IMAGES_DIR / "val" / "*.jpg")))[:50]
print(f"[INFO] Benchmark sur {len(test_images)} images.")

# ── YOLOv5 ───────────────────────────────────────────────────────────────────
v5_model_path = str(YV5_RESULT_DIR / "weights" / "best.pt")
yv5_infer = torch.hub.load(str(YOLOV5_DIR), "custom", path=v5_model_path, source="local", verbose=False)
yv5_infer.eval()
for _ in range(5): yv5_infer(test_images[0])  # warm-up

times_v5 = []
for img_path in test_images:
    t0 = time.perf_counter()
    yv5_infer(img_path)
    times_v5.append((time.perf_counter() - t0) * 1000)
ms_v5  = float(np.mean(times_v5))
std_v5 = float(np.std(times_v5))
fps_v5 = 1000.0 / ms_v5
print(f"[YOLOv5s] {ms_v5:.2f} ± {std_v5:.2f} ms/image — {fps_v5:.1f} FPS")

# ── YOLOv8 ───────────────────────────────────────────────────────────────────
v8_model_path = str(YV8_RESULT_DIR / "weights" / "best.pt")
yv8_infer = YOLO(v8_model_path)
for _ in range(5): yv8_infer(test_images[0], verbose=False)  # warm-up

times_v8 = []
for img_path in test_images:
    t0 = time.perf_counter()
    yv8_infer(img_path, verbose=False)
    times_v8.append((time.perf_counter() - t0) * 1000)
ms_v8  = float(np.mean(times_v8))
std_v8 = float(np.std(times_v8))
fps_v8 = 1000.0 / ms_v8
print(f"[YOLOv8s] {ms_v8:.2f} ± {std_v8:.2f} ms/image — {fps_v8:.1f} FPS")

YOLOv5 🚀 v7.0-509-gfd0f6da7 Python-3.11.10 torch-2.4.1+cu124 CUDA:0 (NVIDIA A40, 45498MiB)



[INFO] Benchmark sur 50 images.


Fusing layers... 
Model summary: 157 layers, 7064065 parameters, 0 gradients, 15.9 GFLOPs
Adding AutoShape... 


[YOLOv5s] 13.65 ms/image — 73.3 FPS
[YOLOv8s] 17.69 ms/image — 56.5 FPS


In [ ]:
import matplotlib.patches as mpatches

models = ["YOLOv5s", "YOLOv8s"]
ms_vals  = [ms_v5,  ms_v8]
fps_vals = [fps_v5, fps_v8]
colors   = ["#2196F3", "#FF5722"]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle("Comparaison de Vitesse d'Inférence", fontsize=14, fontweight="bold")

# ms / image
ax = axes[0]
bars = ax.bar(models, ms_vals, color=colors, width=0.4)
ax.set_title("Temps d'inférence (ms / image)")
ax.set_ylabel("ms")
ax.bar_label(bars, fmt="%.1f ms", padding=3, fontsize=11)
ax.set_ylim(0, max(ms_vals) * 1.3)
ax.grid(axis="y", alpha=0.3)

# FPS
ax = axes[1]
bars = ax.bar(models, fps_vals, color=colors, width=0.4)
ax.set_title("FPS (images par seconde)")
ax.set_ylabel("FPS")
ax.bar_label(bars, fmt="%.1f FPS", padding=3, fontsize=11)
ax.set_ylim(0, max(fps_vals) * 1.3)
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig(RUN_ROOT / "speed_comparison.png", dpi=150)
plt.show()
print(f"[OK] Graphique sauvegardé dans {RUN_ROOT}/speed_comparison.png")

[OK] Graphique sauvegardé : speed_comparison.png


## 18. Tableau récapitulatif final

In [ ]:
import pandas as pd, numpy as np, json
import matplotlib.pyplot as plt

summary = pd.DataFrame({
    "Modèle":               ["YOLOv5s", "YOLOv8s"],
    "mAP50 (final)": [df_v5[V5_MAP50].iloc[-1],   df_v8[V8_MAP50].iloc[-1]],
    "mAP50 (best)":  [df_v5[V5_MAP50].max(),       df_v8[V8_MAP50].max()],
    "mAP50-95":      [df_v5[V5_MAP5095].iloc[-1],  df_v8[V8_MAP5095].iloc[-1]],
    "Précision":     [df_v5[V5_PREC].iloc[-1],     df_v8[V8_PREC].iloc[-1]],
    "Rappel":        [df_v5[V5_REC].iloc[-1],      df_v8[V8_REC].iloc[-1]],
    "F1 (max)":      [f1_v5.max(),                  f1_v8.max()],
    "Inférence (ms)": [ms_v5,  ms_v8],
    "FPS":            [fps_v5, fps_v8],
})

for col in ["mAP50 (final)", "mAP50 (best)", "mAP50-95", "Précision", "Rappel", "F1 (max)"]:
    summary[col] = summary[col].map(lambda x: f"{x*100:.2f}%")
summary["Inférence (ms)"] = summary["Inférence (ms)"].map(lambda x: f"{x:.2f}")
summary["FPS"]            = summary["FPS"].map(lambda x: f"{x:.1f}")

print("=" * 70)
print(f"  RÉSULTATS — Run {TIMESTAMP}")
print("  YOLOv5s vs YOLOv8s sur Pascal VOC 2007")
print("=" * 70)
display(summary.set_index("Modèle"))

# ── Sauvegarder le résumé CSV ─────────────────────────────────────────────────
summary.to_csv(RUN_ROOT / "summary.csv", index=False)
print(f"[OK] Résumé sauvegardé : {RUN_ROOT}/summary.csv")

# ── Radar chart ───────────────────────────────────────────────────────────────
categories = ["mAP50", "mAP50-95", "Précision", "Rappel", "F1", "FPS (norm.)"]

N = len(categories)print("       └── yolov8/voc/  (weights, results.csv ...)")

max_fps = max(fps_v5, fps_v8)print("       ├── yolov5/voc/  (weights, results.csv ...)")

vals_v5 = [df_v5[V5_MAP50].max(), df_v5[V5_MAP5095].max(),print("       ├── radar_comparison.png")

           df_v5[V5_PREC].max(), df_v5[V5_REC].max(), f1_v5.max(), fps_v5 / max_fps]print("       ├── speed_comparison.png")

vals_v8 = [df_v8[V8_MAP50].max(), df_v8[V8_MAP5095].max(),print("       ├── per_class_ap.png")

           df_v8[V8_PREC].max(), df_v8[V8_REC].max(), f1_v8.max(), fps_v8 / max_fps]print("       ├── f1_comparison.png")

print("       ├── precision_recall_comparison.png")

angles  = [n / float(N) * 2 * np.pi for n in range(N)] + [0]print("       ├── loss_comparison.png")

vals_v5 = vals_v5 + vals_v5[:1]print("       ├── map_comparison.png")

vals_v8 = vals_v8 + vals_v8[:1]print("       ├── hyp_advanced.yaml")

print("       ├── summary.csv")

fig, ax = plt.subplots(figsize=(7, 7), subplot_kw={"polar": True})print("       ├── run_info.json")

ax.plot(angles, vals_v5, "o-", linewidth=2, color="#2196F3", label="YOLOv5s")print(f"     Structure : results/ameliored/{TIMESTAMP}/")

ax.fill(angles, vals_v5, alpha=0.15, color="#2196F3")print(f"\n[OK] Tous les résultats sauvegardés dans : {RUN_ROOT}")

ax.plot(angles, vals_v8, "o-", linewidth=2, color="#FF5722", label="YOLOv8s")

ax.fill(angles, vals_v8, alpha=0.15, color="#FF5722")plt.show()

ax.set_xticks(angles[:-1])plt.savefig(RUN_ROOT / "radar_comparison.png", dpi=150)

ax.set_xticklabels(categories, fontsize=10)plt.tight_layout()

ax.set_title(f"Synthèse — Run {TIMESTAMP}", fontsize=12, fontweight="bold", pad=20)ax.grid(True, alpha=0.3)
ax.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1))

  RÉSULTATS COMPARATIFS — YOLOv5s vs YOLOv8s sur Pascal VOC 2007


,mAP50 (final),mAP50-95 (final),Précision (finale),Rappel (final),Inférence (ms),FPS
Modèle,,,,,,
YOLOv5s,99.36%,85.72%,98.72%,98.42%,13.65,73.3
YOLOv8s,99.49%,93.66%,99.46%,99.43%,17.69,56.5


In [23]:
print(v8_model_path)

/workspace/runs/detect/runs/yolov8/voc/weights/best.pt
